# Практика: CPO и ICE

В уроках модуля мы разобрали, как меняется прогноз модели, если менять один признак объекта: кривую CPO для одного объекта, пучок ICE-кривых для набора объектов и два варианта ICE — центрированный (c-ICE) и через производную (d-ICE). В этой тетради все это строится руками на настоящих данных и сверяется с реализацией в `scikit-learn`.

В каждом задании есть ячейка с `...` — ее нужно дописать. Заданий в тренажере по этой тетради нет: проверяйте себя по графикам и по сверке с `sklearn` в ячейках.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence

%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Данные и модель

Набор California Housing встроен в `sklearn`: 20 640 районов Калифорнии из переписи 1990 года, восемь признаков района, целевая переменная — медианная стоимость дома в районе в сотнях тысяч долларов. Признаки: `MedInc` — медианный доход (десятки тысяч долларов), `HouseAge` — медианный возраст дома, `AveRooms` и `AveBedrms` — среднее число комнат и спален на дом, `Population` — население района, `AveOccup` — среднее число жильцов на дом, `Latitude` и `Longitude` — широта и долгота.

Делим данные на обучающую и тестовую части (80% и 20%, `random_state=42`) и обучаем градиентный бустинг `HistGradientBoostingRegressor` с параметрами по умолчанию. Для методов этой тетради модель — черный ящик: от нее нужен только прогноз `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"R^2 модели на тестовых данных: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. CPO: один объект, один признак

Фиксируем объект $x_i$, перебираем значения $z$ признака $j$ по сетке и для каждого значения запрашиваем прогноз $f(x_i^{z})$, где $x_i^{z}$ — объект $x_i$, у которого признак $j$ заменен на $z$.

**Задание 1.** Допишите функцию `cpo_curve`: она размножает объект `x` на `len(grid)` строк, ставит в столбец `feature` значения сетки и возвращает прогнозы модели. Сетку для графиков берем из 20 точек между 5-м и 95-м процентилями признака на обучающих данных.

In [ ]:
def cpo_curve(model, x, feature, grid):
    X_rep = ...  # ваш код: len(grid) копий объекта x, в столбце feature — значения grid
    return ...


def quantile_grid(X, feature, k=20):
    return np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), k)


x0 = X_test.iloc[0]
f0 = model.predict(X_test.iloc[[0]])[0]
print(f"Прогноз модели для объекта x0: {f0:.3f}")
x0.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, feature in zip(axes, ["MedInc", "Latitude"]):
    grid = quantile_grid(X_train, feature)
    ax.plot(grid, cpo_curve(model, x0, feature, grid), color=BLUE, lw=2)
    ax.scatter([x0[feature]], [f0], color=ORANGE, zorder=3, label="сам объект x0")
    ax.set_xlabel(feature)
    ax.legend()
axes[0].set_ylabel("прогноз, сотни тысяч долларов")
fig.suptitle("CPO-кривые объекта x0: California Housing, градиентный бустинг")
plt.show()

## 2. CPO-оценка

$$O_j^{uni}(x_i) = \frac{1}{k}\sum_{l=1}^{k}\bigl| f(x_i^{z_l}) - f(x_i) \bigr|,\qquad O_j^{emp}(x_i) = \frac{1}{n}\sum_{m=1}^{n}\bigl| f(x_i^{z_m}) - f(x_i) \bigr|.$$

В первой оценке $z_l$ — точки равномерной сетки по диапазону признака, во второй $z_m$ — значения признака у $n$ наблюдений выборки.

**Задание 2.** Допишите `o_uni` и `o_emp` и посчитайте обе оценки для объекта `x0` по всем признакам. Равномерную сетку возьмите из $k=20$ точек по всему диапазону признака на обучающих данных, от минимума до максимума; эмпирическую оценку — по значениям признака у 500 случайных строк обучающих данных (`random_state=0`).

In [ ]:
def o_uni(model, x, feature, X_ref, k=20):
    grid = ...  # ваш код: k точек от минимума до максимума признака
    f_x = model.predict(pd.DataFrame([x]))[0]
    return ...


def o_emp(model, x, feature, X_ref):
    f_x = model.predict(pd.DataFrame([x]))[0]
    return ...  # ваш код: среднее модуля отклонения прогноза по значениям признака в X_ref


X_ref = X_train.sample(500, random_state=0)
cpo_scores = pd.DataFrame({
    "O_uni": [o_uni(model, x0, f, X_train) for f in X.columns],
    "O_emp": [o_emp(model, x0, f, X_ref) for f in X.columns],
}, index=X.columns)
cpo_scores.round(3)

**Место для ваших выводов:**
- какой признак сильнее всего раскачивает прогноз объекта `x0` по каждой из двух оценок?
- у каких признаков две оценки расходятся сильнее всего и почему? Подсказка: сравните в `X.describe()` максимум признака с его 75-м процентилем.

## 3. ICE: пучок CPO-кривых

ICE — это CPO-кривые сразу для набора объектов. Возьмем 200 случайных объектов тестовой выборки.

**Задание 3.** Допишите `ice_curves`: функция возвращает матрицу размера `(len(X), len(grid))`, где строка — ICE-кривая одного объекта. Сделайте это одним вызовом `model.predict`: склейте `len(grid)` копий `X` и в каждой копии поставьте в столбец `feature` свое значение сетки. Затем сверьте результат с `sklearn.inspection.partial_dependence(kind='individual')` на той же сетке.

In [ ]:
def ice_curves(model, X, feature, grid):
    X_rep = ...  # ваш код: len(grid) копий X подряд; в копии t столбец feature равен grid[t]
    return ...   # матрица (len(X), len(grid))


X_ice = X_test.sample(200, random_state=0)
sk = partial_dependence(model, X_ice, ["MedInc"], kind="individual", grid_resolution=20, percentiles=(0.05, 0.95))
grid = sk["grid_values"][0]
ice = ice_curves(model, X_ice, "MedInc", grid)
print("Наибольшее расхождение с sklearn:", np.abs(ice - sk["individual"][0]).max())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(grid, ice[:30].T, color=BLUE, alpha=0.6, lw=1)
ax.scatter(np.full(30, grid[0]), ice[:30, 0], color=BLUE, s=12, zorder=3)
ax.set_xlabel("MedInc — медианный доход района, десятки тысяч долларов")
ax.set_ylabel("прогноз, сотни тысяч долларов")
ax.set_title("ICE по MedInc для 30 объектов California Housing, градиентный бустинг")
plt.show()

Кривые стартуют с разных уровней: у каждого района свой прогноз уже при самом низком доходе. Добавим все 200 кривых и их среднее по объектам — это и есть PDP, о котором следующий урок модуля.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(grid, ice.T, color=BLUE, alpha=0.2, lw=0.8)
ax.plot(grid, ice.mean(axis=0), color=ORANGE, lw=3, label="среднее ICE-кривых (PDP)")
ax.set_xlabel("MedInc — медианный доход района, десятки тысяч долларов")
ax.set_ylabel("прогноз, сотни тысяч долларов")
ax.set_title("ICE по MedInc для 200 объектов California Housing, градиентный бустинг")
ax.legend()
plt.show()

Если объектов тысячи, пучок сливается в сплошную полосу: отдельные траектории уже не прочитать, видно только среднее и общий разброс. Построим ICE по признаку `HouseAge` для всех объектов тестовой выборки.

In [ ]:
g_age = quantile_grid(X_train, "HouseAge")
ice_all = ice_curves(model, X_test, "HouseAge", g_age)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(g_age, ice_all.T, color=BLUE, alpha=0.08, lw=0.6)
ax.plot(g_age, ice_all.mean(axis=0), color=ORANGE, lw=3, label="среднее")
ax.set_xlabel("HouseAge — медианный возраст дома в районе, лет")
ax.set_ylabel("прогноз, сотни тысяч долларов")
ax.set_title(f"ICE по HouseAge для всех {len(X_test)} объектов тестовой выборки, градиентный бустинг")
ax.legend()
plt.show()

## 4. c-ICE

$$g_i^{c}(x_j) = g_i(x_j) - g_i(a),$$

где опорная точка $a$ — первое значение сетки.

**Задание 4.** Допишите `center` и постройте c-ICE для четырех признаков: `MedInc`, `AveOccup`, `HouseAge`, `Latitude`. Поверх кривых проведите их среднее. Посчитайте разброс центрированных кривых в последней точке сетки — стандартное отклонение по 200 объектам.

In [ ]:
def center(ice):
    return ...  # ваш код: вычесть из каждой кривой ее значение в первой точке сетки


features = ["MedInc", "AveOccup", "HouseAge", "Latitude"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
spread_end = {}
for ax, feature in zip(axes.ravel(), features):
    g = quantile_grid(X_train, feature)
    c = center(ice_curves(model, X_ice, feature, g))
    spread_end[feature] = c[:, -1].std()
    ax.plot(g, c.T, color=BLUE, alpha=0.2, lw=0.8)
    ax.plot(g, c.mean(axis=0), color=ORANGE, lw=3)
    ax.set_xlabel(feature)
    ax.set_ylabel("прогноз минус прогноз в точке a")
fig.suptitle("c-ICE для 200 объектов California Housing, градиентный бустинг; оранжевая линия — среднее")
fig.tight_layout()
plt.show()
pd.Series(spread_end, name="разброс c-ICE в последней точке").round(3)

**Место для ваших выводов:** у какого из четырех признаков центрированные кривые расходятся сильнее всего, то есть эффект признака сильнее всего зависит от остальных признаков района? У какого слабее всего?

## 5. d-ICE

Производную у бустинга взять нельзя: прогноз деревьев кусочно-постоянен. Поэтому наклон считаем конечной разностью на отрезках сетки:

$$g_i'(x_j) \approx \frac{g_i(z_{l+1}) - g_i(z_l)}{z_{l+1} - z_l}.$$

**Задание 5.** Допишите `d_ice` и для каждого из четырех признаков найдите отрезок сетки, на котором наклоны разных объектов расходятся сильнее всего: максимум стандартного отклонения наклонов по объектам.

In [ ]:
def d_ice(ice, grid):
    return ...  # ваш код: матрица наклонов (len(ice), len(grid) - 1)


rows = {}
for feature in features:
    g = quantile_grid(X_train, feature)
    s = d_ice(ice_curves(model, X_ice, feature, g), g).std(axis=0)
    l = s.argmax()
    rows[feature] = {"наибольший разброс наклонов": s.max(),
                     "отрезок от": g[l], "до": g[l + 1]}
pd.DataFrame(rows).T.round(3)

**Место для ваших выводов:** на каком отрезке и у какого признака взаимодействие с остальными признаками сильнее всего? Сравните с c-ICE из задания 4.

## 6. ICE линейной модели

Обучим на тех же данных линейную регрессию и построим ее ICE-кривые по признаку `HouseAge` для 40 объектов.

**Задание 6.** Проверьте две вещи: наклоны d-ICE одинаковы у всех объектов, и этот общий наклон равен весу признака `HouseAge` в модели (`lin.coef_`).

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
g = quantile_grid(X_train, "HouseAge")
ice_lin = ice_curves(lin, X_ice.iloc[:40], "HouseAge", g)
slopes_lin = d_ice(ice_lin, g)
w = ...  # ваш код: вес признака HouseAge в линейной модели
print("Разброс наклонов по объектам:", slopes_lin.std(axis=0).max())
print("Наклон:", slopes_lin[0, 0], " вес:", w)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(g, ice_lin.T, color=BLUE, alpha=0.5, lw=1)
ax.plot(g, ice_lin.mean(axis=0), color=ORANGE, lw=2.5, label="среднее")
ax.set_xlabel("HouseAge — медианный возраст дома в районе, лет")
ax.set_ylabel("прогноз, сотни тысяч долларов")
ax.set_title("ICE по HouseAge для 40 объектов California Housing")
ax.legend()
plt.show()

**Место для ваших выводов:** почему у линейной модели ICE-кривые параллельны и что из этого следует для c-ICE и d-ICE?